
### Fase 7: Performance





[Voltar para notebook principal](./0_notebook_principal.ipynb)

In [10]:
import time
import torch
import gc
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pathlib import Path

# --- Configurações ---
BASE_DIR = Path.cwd()
STUDENT_MODEL_PATH = BASE_DIR / "models" / "student_model"
TEACHER_MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
OUTPUT_DIR = BASE_DIR / "reports" / "figures"
DEVICE = "cpu" # Forçamos CPU para evitar travamentos, ou "cuda" se tiver certeza

def measure_model_performance(model_path_or_name, model_alias, sample_texts):
    print(f"   🔄 Carregando {model_alias}...")
    
    # 1. Carrega Modelo e Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path_or_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_path_or_name)
    model.to(DEVICE)
    model.eval() # Modo de avaliação (mais leve)

    # 2. Prepara Input
    inputs = tokenizer(sample_texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)

    # 3. Mede o Tempo (Warmup + Loop)
    print(f"   ⏱️ Medindo latência do {model_alias}...")
    
    # Aquecimento (Warmup) - para carregar caches
    with torch.no_grad():
        _ = model(**inputs)
    
    # Medição Real - múltiplas repetições
    N = 100
    start_time = time.time()
    with torch.no_grad():
        for _ in range(N):
            _ = model(**inputs)
    end_time = time.time()

    duration = (end_time - start_time) / N  # média por inferência
    print(f"   ✅ {model_alias}: {duration:.4f} segundos")

    # 4. LIMPEZA CRÍTICA DE MEMÓRIA
    del model
    del tokenizer
    del inputs
    gc.collect() # Força o Python a liberar RAM
    if torch.cuda.is_available():
        torch.cuda.empty_cache() # Força GPU a liberar VRAM
        
    return duration

def run_safe_benchmark():
    print("--- ⚡ Benchmark Seguro de Performance (Sequencial) ---")
    
    # Criar uma amostra de texto (100 frases iguais para teste de carga)
    sample_texts = ["I am extremely dissatisfied with the service provided by the bank."] * 50
    
    # 1. Medir Student (DistilBERT)
    student_time = measure_model_performance(STUDENT_MODEL_PATH, "Student (DistilBERT)", sample_texts)
    
    # 2. Medir Teacher (RoBERTa)
    # Nota: Se o Teacher for muito pesado, reduzimos a amostra ou aceitamos que ele é lento
    teacher_time = measure_model_performance(TEACHER_MODEL_NAME, "Teacher (RoBERTa)", sample_texts)
    
    # 3. Plotar Resultados
    print("\n📊 Gerando Gráfico Comparativo...")
    times = [teacher_time, student_time]
    labels = ['Teacher\n(RoBERTa)', 'Student\n(DistilBERT)']
    colors = ['#ff9999', '#66b3ff'] # Vermelho (Lento), Azul (Rápido)
    
    plt.figure(figsize=(8, 6))
    bars = plt.bar(labels, times, color=colors)
    
    plt.ylabel('Tempo de Inferência (segundos)')
    plt.title(f'Benchmark de Velocidade (Processando {len(sample_texts)} frases)')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Calcular Speedup (Quantas vezes mais rápido)
    speedup = teacher_time / student_time
    
    # Adicionar texto nas barras
    for bar, time_val in zip(bars, times):
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + (max(times)*0.01), f"{time_val:.2f}s", ha='center', va='bottom')

    # Texto de destaque
    plt.text(1, student_time / 2, f"{speedup:.1f}x mais rápido!", 
             ha='center', va='center', color='white', fontweight='bold', fontsize=12,
             bbox=dict(facecolor='black', alpha=0.5))
    
    output_path = OUTPUT_DIR / "latency_benchmark_safe.png"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path)
    plt.close()
    
    print(f"✅ Gráfico salvo em: {output_path}")
    print(f"🚀 Conclusão: O modelo Student é {speedup:.2f}x mais rápido que o Teacher.")

if __name__ == "__main__":
    run_safe_benchmark()

--- ⚡ Benchmark Seguro de Performance (Sequencial) ---
   🔄 Carregando Student (DistilBERT)...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

   ⏱️ Medindo latência do Student (DistilBERT)...
   ✅ Student (DistilBERT): 0.1118 segundos
   🔄 Carregando Teacher (RoBERTa)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   ⏱️ Medindo latência do Teacher (RoBERTa)...
   ✅ Teacher (RoBERTa): 0.2337 segundos

📊 Gerando Gráfico Comparativo...
✅ Gráfico salvo em: F:\workspace\postech\datathon_tech_challenge_5\reports\figures\latency_benchmark_safe.png
🚀 Conclusão: O modelo Student é 2.09x mais rápido que o Teacher.



[Voltar para notebook principal](./0_notebook_principal.ipynb)